In [1]:
# ── Walk‑Forward Kelly Simulation (de‑vigged features for inference) ───────
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════
# 0. Power‑method de‑vigging (same as app & original simulation)
# ═══════════════════════════════════════════════════════════════════════════
def power_de_vig(home_implied, away_implied, max_iter=100, tol=1e-12):
    """
    Remove the overround using the power (multiplicative) method.
    Finds a scalar k such that:
        home_implied^k + away_implied^k = 1
    and returns fair_home, fair_away.
    """
    home_implied = np.asarray(home_implied, dtype=float)
    away_implied = np.asarray(away_implied, dtype=float)
    n = len(home_implied)
    fair_home = np.full(n, np.nan)
    fair_away = np.full(n, np.nan)

    for i in range(n):
        h = home_implied[i]
        a = away_implied[i]
        if np.isnan(h) or np.isnan(a) or h <= 0 or a <= 0:
            continue
        if abs(h + a - 1.0) < tol:
            fair_home[i], fair_away[i] = h, a
            continue

        def f(k):
            return h**k + a**k - 1.0
        def fp(k):
            if h > 0 and a > 0:
                return h**k * np.log(h) + a**k * np.log(a)
            return 0.0

        k = 1.0
        converged = False
        for _ in range(max_iter):
            fk = f(k)
            if abs(fk) < tol:
                converged = True
                break
            fpk = fp(k)
            if fpk == 0:
                break
            k_new = k - fk / fpk
            if k_new <= 0: k_new = 0.001
            if k_new > 10.0: k_new = 10.0
            if abs(k_new - k) < tol:
                k = k_new; converged = True; break
            k = k_new

        if not converged:
            lo, hi = 0.001, 1.0
            if f(lo) < 0: lo, hi = hi, lo
            for _ in range(max_iter):
                mid = (lo + hi) / 2
                if f(mid) == 0.0 or (hi - lo) / 2 < tol:
                    k = mid; converged = True; break
                if np.sign(f(mid)) == np.sign(f(lo)):
                    lo = mid
                else:
                    hi = mid
            if not converged:
                total = h + a
                fair_home[i] = h / total if total > 0 else np.nan
                fair_away[i] = a / total if total > 0 else np.nan
                continue

        fair_home[i] = h**k
        fair_away[i] = a**k
    return fair_home, fair_away

# ═══════════════════════════════════════════════════════════════════════════
# 1. Load model bundle & data
# ═══════════════════════════════════════════════════════════════════════════
bundle = joblib.load("nba.pkl")
model = bundle["model"]
scaler = bundle["scaler"]
selected_features = bundle["features"]

INITIAL_BANKROLL = 10000.0
MAX_DRAWDOWN_LIMIT = -0.70          # -70%

df = pd.read_csv("../data/csv/dataset.csv")

# ═══════════════════════════════════════════════════════════════════════════
# 2. Feature engineering (raw → de‑vigged, EXACTLY like the app)
# ═══════════════════════════════════════════════════════════════════════════
def feature_engineering_app_style(df):
    """Replicates the app's inference pipeline:
       1. Parse odds to raw implied prob.
       2. De‑vig using power method (overwrites raw implied probs).
       3. Build diff/ratio features from the FAIR probs.
    """
    df = df.copy()

    # ---- Parse to raw implied prob (same as original) ----
    for col in ("home_moneyline", "away_moneyline"):
        if col not in df.columns:
            continue
        clean = (df[col].astype(str)
                 .str.replace("+", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .str.replace(" ", "", regex=False))
        num = pd.to_numeric(clean, errors="coerce")
        implied = np.where(
            num >= 100, 100 / (num + 100),
            np.where(
                num <= -100,
                np.abs(num) / (np.abs(num) + 100),
                np.where(
                    (num > 1.0) & (num < 100.0),
                    1.0 / num,
                    np.nan
                )
            )
        )
        df[f"{col}_implied_prob"] = implied

    # ---- De‑vig BEFORE feature construction (exactly like the app) ----
    col_h = "home_moneyline_implied_prob"
    col_a = "away_moneyline_implied_prob"
    if col_h in df.columns and col_a in df.columns:
        # Apply row‑by‑row power de‑vigging
        h_vals = df[col_h].values
        a_vals = df[col_a].values
        fair_h, fair_a = power_de_vig(h_vals, a_vals)
        df[col_h] = fair_h
        df[col_a] = fair_a

    # ---- Derived features from FAIR probs ----
    req = ("home_moneyline_implied_prob", "away_moneyline_implied_prob")
    if all(c in df.columns for c in req):
        df["prob_diff"] = df[req[0]] - df[req[1]]
        df["prob_ratio"] = df[req[0]] / (df[req[1]] + 1e-5)

    if "home_elo" in df.columns and "away_elo" in df.columns:
        df["elo_diff"] = df["home_elo"] - df["away_elo"]
        df["elo_ratio"] = df["home_elo"] / (df["away_elo"] + 1e-5)

    return df

df = feature_engineering_app_style(df)

# Drop rows where both moneylines are missing (keep as many as possible)
df = df.dropna(subset=["home_moneyline_implied_prob", "away_moneyline_implied_prob"]).reset_index(drop=True)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

# ═══════════════════════════════════════════════════════════════════════════
# 3. Generate candidate strategies (identical to earlier search)
# ═══════════════════════════════════════════════════════════════════════════
def generate_strategies():
    strategies = []
    min_edge_options = [0.0, 0.01, 0.02, 0.025, 0.03,0.035, 0.04]
    kelly_fracs = [0.25, 0.375, 0.5, 0.75, 1.0]
    cutoffs = [0.005 + 0.01 * i for i in range(10)]

    # Flat
    for min_e in min_edge_options:
        for f in kelly_fracs:
            name = f"Flat_min{min_e:.3f}_kelly{f}"
            strategies.append((name, [(min_e, np.inf, f)]))

    # Two‑bin
    for min_e in min_edge_options:
        for thr in cutoffs:
            if thr <= min_e: continue
            for f_low in kelly_fracs:
                for f_high in kelly_fracs:
                    name = (f"2Bin_min{min_e:.3f}_lo<{thr:.3f}_"
                            f"fL{f_low}_fH{f_high}")
                    strategies.append((name, [
                        (min_e, thr, f_low),
                        (thr, np.inf, f_high)
                    ]))

    # Three‑bin
    breakpoint_pairs = [
        (0.005, 0.025), (0.015, 0.035), (0.025, 0.045), (0.035, 0.055),
        (0.005, 0.035), (0.015, 0.045), (0.025, 0.055), (0.035, 0.065),
    ]
    for min_e in [0.0, 0.01, 0.02]:
        for (bp1, bp2) in breakpoint_pairs:
            if bp1 <= min_e or bp2 <= bp1: continue
            for f1 in [0.25, 0.375, 0.5]:
                for f2 in [0.25, 0.375, 0.5, 0.75]:
                    for f3 in [0.25, 0.375, 0.5, 0.75, 1.0]:
                        name = (f"3Bin_min{min_e:.3f}_"
                                f"[{min_e:.3f}-{bp1:.3f}]f{f1}_"
                                f"[{bp1:.3f}-{bp2:.3f}]f{f2}_"
                                f"[{bp2:.3f}+]f{f3}")
                        strategies.append((name, [
                            (min_e, bp1, f1),
                            (bp1, bp2, f2),
                            (bp2, np.inf, f3)
                        ]))
    return strategies

strategies = generate_strategies()
print(f"Generated {len(strategies)} strategies.\n")

# ═══════════════════════════════════════════════════════════════════════════
# 4. Simulation function (Kelly betting, uses de‑vigged edge calculation)
#    Note: test_df already contains DE‑VIGGED implied probs.
#    So edge = model_prob – (de‑vigged implied prob) = model_prob – fair_prob.
#    This matches the app's calculation of edge.
# ═══════════════════════════════════════════════════════════════════════════
def simulate_kelly(test_df, y_pred_proba, bankroll, edge_strategy):
    df = test_df.copy()
    df["model_prob_away"] = y_pred_proba
    df["model_prob_home"] = 1 - y_pred_proba

    # Fair prob is already the de‑vigged value stored in the columns
    df["fair_home_prob"] = df["home_moneyline_implied_prob"]
    df["fair_away_prob"] = df["away_moneyline_implied_prob"]

    df["home_edge"] = df["model_prob_home"] - df["fair_home_prob"]
    df["away_edge"] = df["model_prob_away"] - df["fair_away_prob"]

    current = bankroll
    bank_ts = [bankroll]
    bets = []

    for idx, row in df.iterrows():
        side = None
        if row["home_edge"] > 0:
            side, edge, imp = "home", row["home_edge"], row["home_moneyline_implied_prob"]
        elif row["away_edge"] > 0:
            side, edge, imp = "away", row["away_edge"], row["away_moneyline_implied_prob"]
        else:
            bank_ts.append(current)
            continue

        if imp <= 0 or np.isnan(imp):
            bank_ts.append(current)
            continue

        # Determine Kelly fraction from strategy
        kelly_frac = None
        for low, high, frac in edge_strategy:
            if low <= edge < high:
                kelly_frac = frac
                break
        if kelly_frac is None:
            bank_ts.append(current)
            continue

        decimal_odds = 1.0 / imp
        p = row["model_prob_home"] if side == "home" else row["model_prob_away"]
        f_full = (p * decimal_odds - 1) / (decimal_odds - 1)
        f = max(0.0, min(f_full * kelly_frac, 1.0))
        stake = f * current
        if stake <= 0:
            bank_ts.append(current)
            continue

        won = (row["winning_team"] == 0) if side == "home" else (row["winning_team"] == 1)
        profit = stake * (decimal_odds - 1) if won else -stake
        current += profit
        bank_ts.append(current)
        bets.append({
            "side": side, "edge": edge, "kelly_frac": kelly_frac,
            "stake": stake, "profit": profit, "won": won
        })

    if not bets:
        return {"final_bankroll": current, "num_bets": 0, "roi_pct": 0.0,
                "win_rate": 0.0, "max_drawdown": 0.0}

    bet_df = pd.DataFrame(bets)
    roi = ((current - bankroll) / bankroll) * 100.0
    win_rate = bet_df["won"].mean()
    bank_series = pd.Series(bank_ts)
    running_max = bank_series.cummax()
    drawdown = (bank_series - running_max) / running_max
    max_dd = drawdown.min()

    return {
        "final_bankroll": round(current, 2),
        "roi_pct": round(roi, 2),
        "num_bets": len(bet_df),
        "home_bets": (bet_df["side"] == "home").sum(),
        "away_bets": (bet_df["side"] == "away").sum(),
        "win_rate": win_rate,
        "mean_edge": bet_df["edge"].mean(),
        "max_drawdown": round(max_dd, 4),
    }

# ═══════════════════════════════════════════════════════════════════════════
# 5. Walk‑forward over multiple splits using DE‑VIGGED FEATURES
# ═══════════════════════════════════════════════════════════════════════════
SPLITS = [0.85, 0.90, 0.95]
all_results = {}   # (split, strategy_name) -> result dict
per_split_top = {}

for split in SPLITS:
    print(f"Processing split = {split:.2f} ...")
    split_idx = int(len(df) * split)
    test_df = df.iloc[split_idx:].reset_index(drop=True)

    # Build feature matrix using de‑vigged features (exactly as the app does)
    COLS_TO_DROP = [
        "winning_team", "date", "home_team", "away_team",
        "home_moneyline", "away_moneyline",
    ]
    X = test_df.drop(columns=[c for c in COLS_TO_DROP if c in test_df.columns])
    X = X.select_dtypes(include=[np.number]).fillna(0)
    X = X.reindex(columns=scaler.feature_names_in_, fill_value=0)
    X_scaled = pd.DataFrame(
        scaler.transform(X),
        columns=X.columns,
        index=X.index
    )
    X_final = X_scaled[selected_features]

    # Model prediction (on de‑vigged features!)
    y_pred_proba = model.predict_proba(X_final)[:, 1]

    # Simulate every strategy on this split
    split_res = {}
    for name, edge_strat in strategies:
        sim = simulate_kelly(test_df, y_pred_proba, INITIAL_BANKROLL, edge_strat)
        if sim["num_bets"] > 0:
            split_res[name] = sim
            all_results[(split, name)] = sim

    # Best for this split (subject to drawdown)
    passed = {n: r for n, r in split_res.items() if r["max_drawdown"] >= MAX_DRAWDOWN_LIMIT}
    if passed:
        top_name, top_res = max(passed.items(), key=lambda x: x[1]["roi_pct"])
        per_split_top[split] = (top_name, top_res)
    else:
        per_split_top[split] = None
    print(f"  -> Best on split {split:.2f}: {per_split_top[split]}\n")

# ═══════════════════════════════════════════════════════════════════════════
# 6. Robust selection: must pass drawdown limit on EVERY split
# ═══════════════════════════════════════════════════════════════════════════
strategy_names = [name for name, _ in strategies]
robust_scores = {}
for name in strategy_names:
    splits_data = {}
    present_all = True
    for split in SPLITS:
        if (split, name) in all_results:
            splits_data[split] = all_results[(split, name)]
        else:
            present_all = False
            break
    if not present_all:
        continue
    dd_ok = all(splits_data[split]["max_drawdown"] >= MAX_DRAWDOWN_LIMIT for split in SPLITS)
    if not dd_ok:
        continue
    mean_roi = np.mean([splits_data[s]["roi_pct"] for s in SPLITS])
    robust_scores[name] = mean_roi

if not robust_scores:
    print("No strategy passed the drawdown limit on all splits simultaneously.")
else:
    best_strategy_name = max(robust_scores, key=robust_scores.get)
    best_mean_roi = robust_scores[best_strategy_name]

    print("\n" + "=" * 100)
    print("            ROBUST STRATEGY (de‑vigged features for inference)")
    print("=" * 100)
    print(f"Strategy: {best_strategy_name}")
    for s in SPLITS:
        r = all_results[(s, best_strategy_name)]
        print(f"  Split {s:.2f} | ROI: {r['roi_pct']:+.2f}%  MaxDD: {r['max_drawdown']:.2%}  "
              f"Bets: {r['num_bets']}  Win: {r['win_rate']:.2%}  "
              f"Bank: ${r['final_bankroll']:,.2f}")
    print(f"  Mean ROI across splits: {best_mean_roi:+.2f}%")

    # Top‑5 robust strategies for reference
    top_robust = sorted(robust_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop‑5 robust strategies (by mean ROI):")
    for rank, (name, mean_roi) in enumerate(top_robust, 1):
        print(f"  {rank}. {name}  (mean ROI = {mean_roi:+.2f}%)")

# Print per‑split winners for comparison
print("\n" + "=" * 100)
print("            BEST STRATEGY FOR EACH INDIVIDUAL SPLIT")
print("=" * 100)
for split, data in per_split_top.items():
    if data is not None:
        name, res = data
        print(f"Split {split:.2f}: {name}  ROI={res['roi_pct']:+.2f}%  "
              f"MaxDD={res['max_drawdown']:.2%}  Bets={res['num_bets']}")
    else:
        print(f"Split {split:.2f}: no strategy passed drawdown limit.")

Generated 2440 strategies.

Processing split = 0.85 ...
  -> Best on split 0.85: ('2Bin_min0.035_lo<0.055_fL1.0_fH0.25', {'final_bankroll': 32089.57, 'roi_pct': 220.9, 'num_bets': 334, 'home_bets': np.int64(192), 'away_bets': np.int64(142), 'win_rate': np.float64(0.38922155688622756), 'mean_edge': np.float64(0.05744149619860935), 'max_drawdown': -0.5502})

Processing split = 0.90 ...
  -> Best on split 0.90: ('2Bin_min0.035_lo<0.055_fL1.0_fH0.25', {'final_bankroll': 23743.98, 'roi_pct': 137.44, 'num_bets': 220, 'home_bets': np.int64(133), 'away_bets': np.int64(87), 'win_rate': np.float64(0.38636363636363635), 'mean_edge': np.float64(0.057438079393626544), 'max_drawdown': -0.5134})

Processing split = 0.95 ...
  -> Best on split 0.95: ('2Bin_min0.025_lo<0.055_fL1.0_fH0.25', {'final_bankroll': 57597.17, 'roi_pct': 475.97, 'num_bets': 170, 'home_bets': np.int64(95), 'away_bets': np.int64(75), 'win_rate': np.float64(0.4117647058823529), 'mean_edge': np.float64(0.0494468191572599), 'max_dra

In [2]:
# ── Cross‑evaluation of the three best strategies (de‑vigged features) ────
import numpy as np
import pandas as pd
import joblib

# ═══════════════════════════════════════════════════════════════════════════
# 0. Power‑method de‑vigging (same as everywhere)
# ═══════════════════════════════════════════════════════════════════════════
def power_de_vig(home_implied, away_implied, max_iter=100, tol=1e-12):
    home_implied = np.asarray(home_implied, dtype=float)
    away_implied = np.asarray(away_implied, dtype=float)
    n = len(home_implied)
    fair_home = np.full(n, np.nan)
    fair_away = np.full(n, np.nan)
    for i in range(n):
        h = home_implied[i]
        a = away_implied[i]
        if np.isnan(h) or np.isnan(a) or h <= 0 or a <= 0:
            continue
        if abs(h + a - 1.0) < tol:
            fair_home[i], fair_away[i] = h, a
            continue
        def f(k): return h**k + a**k - 1.0
        def fp(k): return h**k * np.log(h) + a**k * np.log(a) if h>0 and a>0 else 0.0
        k = 1.0
        for _ in range(max_iter):
            fk = f(k)
            if abs(fk) < tol: break
            fpk = fp(k)
            if fpk == 0: break
            k_new = k - fk / fpk
            if k_new <= 0: k_new = 0.001
            if k_new > 10: k_new = 10.0
            if abs(k_new - k) < tol:
                k = k_new; break
            k = k_new
        else:
            lo, hi = 0.001, 1.0
            if f(lo) < 0: lo, hi = hi, lo
            for _ in range(max_iter):
                mid = (lo+hi)/2
                if f(mid)==0 or (hi-lo)/2<tol:
                    k = mid; break
                if np.sign(f(mid))==np.sign(f(lo)): lo = mid
                else: hi = mid
            else:
                total = h + a
                fair_home[i] = h/total if total>0 else np.nan
                fair_away[i] = a/total if total>0 else np.nan
                continue
        fair_home[i] = h**k
        fair_away[i] = a**k
    return fair_home, fair_away

# ═══════════════════════════════════════════════════════════════════════════
# 1. Load model bundle & data
# ═══════════════════════════════════════════════════════════════════════════
bundle = joblib.load("nba.pkl")
model = bundle["model"]
scaler = bundle["scaler"]
selected_features = bundle["features"]

INITIAL_BANKROLL = 10000.0

df = pd.read_csv("../data/csv/dataset.csv")

# Feature engineering: de‑vigging BEFORE inference (exactly as the simulation)
def feature_engineering_app_style(df):
    df = df.copy()
    # Parse raw implied probs
    for col in ("home_moneyline", "away_moneyline"):
        if col not in df.columns:
            continue
        clean = (df[col].astype(str)
                 .str.replace("+", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .str.replace(" ", "", regex=False))
        num = pd.to_numeric(clean, errors="coerce")
        implied = np.where(
            num >= 100, 100 / (num + 100),
            np.where(num <= -100, np.abs(num) / (np.abs(num) + 100),
                     np.where((num > 1.0) & (num < 100.0), 1.0 / num, np.nan))
        )
        df[f"{col}_implied_prob"] = implied

    # De‑vig the implied probs
    col_h, col_a = "home_moneyline_implied_prob", "away_moneyline_implied_prob"
    if col_h in df.columns and col_a in df.columns:
        h_vals = df[col_h].values
        a_vals = df[col_a].values
        fair_h, fair_a = power_de_vig(h_vals, a_vals)
        df[col_h] = fair_h
        df[col_a] = fair_a

    # Derived features from fair probs
    req = ("home_moneyline_implied_prob", "away_moneyline_implied_prob")
    if all(c in df.columns for c in req):
        df["prob_diff"]  = df[req[0]] - df[req[1]]
        df["prob_ratio"] = df[req[0]] / (df[req[1]] + 1e-5)
    if "home_elo" in df.columns and "away_elo" in df.columns:
        df["elo_diff"]  = df["home_elo"] - df["away_elo"]
        df["elo_ratio"] = df["home_elo"] / (df["away_elo"] + 1e-5)
    return df

df = feature_engineering_app_style(df)
df = df.dropna(subset=["home_moneyline_implied_prob", "away_moneyline_implied_prob"])
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

# ═══════════════════════════════════════════════════════════════════════════
# 2. Define the three strategies to test
# ═══════════════════════════════════════════════════════════════════════════
strategies_to_test = {
    "0.85_Best": [
        (0.035, 0.055, 1.0),
        (0.055, np.inf, 0.25)
    ],
    "0.90_Best": [
        (0.035, 0.055, 1.0),
        (0.055, np.inf, 0.25)
    ],
    "0.95_Best": [
        (0.025, 0.055, 1.0),
        (0.055, np.inf, 0.25)
    ]
}

# ═══════════════════════════════════════════════════════════════════════════
# 3. Simulation function (exactly as the robust search)
# ═══════════════════════════════════════════════════════════════════════════
def simulate_kelly(test_df, y_pred_proba, bankroll, edge_strategy):
    df = test_df.copy()
    df["model_prob_away"] = y_pred_proba
    df["model_prob_home"] = 1 - y_pred_proba

    # The implied prob columns already hold the de‑vigged fair probabilities
    df["fair_home_prob"] = df["home_moneyline_implied_prob"]
    df["fair_away_prob"] = df["away_moneyline_implied_prob"]

    df["home_edge"] = df["model_prob_home"] - df["fair_home_prob"]
    df["away_edge"] = df["model_prob_away"] - df["fair_away_prob"]

    current = bankroll
    bank_ts = [bankroll]
    bets = []

    for idx, row in df.iterrows():
        if row["home_edge"] > 0:
            side, edge, imp = "home", row["home_edge"], row["home_moneyline_implied_prob"]
        elif row["away_edge"] > 0:
            side, edge, imp = "away", row["away_edge"], row["away_moneyline_implied_prob"]
        else:
            bank_ts.append(current)
            continue
        if imp <= 0 or np.isnan(imp):
            bank_ts.append(current)
            continue

        kelly_frac = None
        for low, high, frac in edge_strategy:
            if low <= edge < high:
                kelly_frac = frac
                break
        if kelly_frac is None:
            bank_ts.append(current)
            continue

        decimal_odds = 1.0 / imp
        p = row["model_prob_home"] if side == "home" else row["model_prob_away"]
        f_full = (p * decimal_odds - 1) / (decimal_odds - 1)
        f = max(0.0, min(f_full * kelly_frac, 1.0))
        stake = f * current
        if stake <= 0:
            bank_ts.append(current)
            continue

        won = (row["winning_team"] == 0) if side == "home" else (row["winning_team"] == 1)
        profit = stake * (decimal_odds - 1) if won else -stake
        current += profit
        bank_ts.append(current)
        bets.append({"side": side, "edge": edge, "kelly_frac": kelly_frac,
                     "stake": stake, "profit": profit, "won": won})

    if not bets:
        return {"final_bankroll": current, "num_bets": 0, "roi_pct": 0.0,
                "win_rate": 0.0, "max_drawdown": 0.0}

    bet_df = pd.DataFrame(bets)
    roi = ((current - bankroll) / bankroll) * 100.0
    win_rate = bet_df["won"].mean()
    bank_series = pd.Series(bank_ts)
    running_max = bank_series.cummax()
    drawdown = (bank_series - running_max) / running_max
    max_dd = drawdown.min()

    return {
        "final_bankroll": round(current, 2),
        "roi_pct": round(roi, 2),
        "num_bets": len(bet_df),
        "win_rate": win_rate,
        "max_drawdown": round(max_dd, 4),
    }

# ═══════════════════════════════════════════════════════════════════════════
# 4. Run cross‑evaluation
# ═══════════════════════════════════════════════════════════════════════════
SPLITS = [0.85, 0.90, 0.95]
results_grid = {}   # (split, strat_name) -> result dict

for split in SPLITS:
    split_idx = int(len(df) * split)
    test_df = df.iloc[split_idx:].reset_index(drop=True)

    # Build feature matrix (de‑vigged features)
    COLS_DROP = ["winning_team","date","home_team","away_team",
                 "home_moneyline","away_moneyline"]
    X = test_df.drop(columns=[c for c in COLS_DROP if c in test_df.columns])
    X = X.select_dtypes(include=[np.number]).fillna(0)
    X = X.reindex(columns=scaler.feature_names_in_, fill_value=0)
    X_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns, index=X.index)
    X_final = X_scaled[selected_features]
    y_proba = model.predict_proba(X_final)[:, 1]

    for strat_name, edge_strat in strategies_to_test.items():
        res = simulate_kelly(test_df, y_proba, INITIAL_BANKROLL, edge_strat)
        results_grid[(split, strat_name)] = res

# ═══════════════════════════════════════════════════════════════════════════
# 5. Print comparison table
# ═══════════════════════════════════════════════════════════════════════════
print(f"Cross‑evaluation of strategies on all splits (bankroll = ${INITIAL_BANKROLL:,.0f})\n")
header = f"{'Strategy':<12} {'Split':<6} {'ROI':>8} {'MaxDD':>8} {'#Bets':>6} {'Win%':>7} {'Final Bankroll':>13}"
print(header)
print("-" * len(header))

for strat_name in strategies_to_test:
    for split in SPLITS:
        r = results_grid[(split, strat_name)]
        if r["num_bets"] == 0:
            print(f"{strat_name:<12} {split:0.2f}   {'No bets':>8}")
        else:
            print(f"{strat_name:<12} {split:0.2f}   {r['roi_pct']:>+7.2f}% {r['max_drawdown']:>7.2%} "
                  f"{r['num_bets']:>6} {r['win_rate']:>6.2%} ${r['final_bankroll']:>12,.2f}")
    print()

Cross‑evaluation of strategies on all splits (bankroll = $10,000)

Strategy     Split       ROI    MaxDD  #Bets    Win% Final Bankroll
-------------------------------------------------------------------
0.85_Best    0.85   +220.90% -55.02%    334 38.92% $   32,089.57
0.85_Best    0.90   +137.44% -51.34%    220 38.64% $   23,743.98
0.85_Best    0.95   +251.22% -50.00%    116 39.66% $   35,122.33

0.90_Best    0.85   +220.90% -55.02%    334 38.92% $   32,089.57
0.90_Best    0.90   +137.44% -51.34%    220 38.64% $   23,743.98
0.90_Best    0.95   +251.22% -50.00%    116 39.66% $   35,122.33

0.95_Best    0.85   +365.97% -82.95%    505 39.60% $   46,597.26
0.95_Best    0.90    +79.61% -74.88%    327 38.84% $   17,961.04
0.95_Best    0.95   +475.97% -51.92%    170 41.18% $   57,597.17

